In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
import wrds

In [3]:
db = wrds.Connection()

Loading library list...
Done


In [4]:
query = """
SELECT
    permno,
    date,
    ret
FROM crsp.msf
WHERE date >= '1956-01-01'
  AND date <= '1967-12-31'
"""

In [5]:
msf_5767 = db.raw_sql(query)

db.close()

In [6]:
msf_5767

,permno,date,ret
0,10006,1956-01-31,-0.04428
1,10014,1956-01-31,0.083333
2,10022,1956-01-31,0.0
3,10030,1956-01-31,-0.015337
4,10057,1956-01-31,-0.037313
...,...,...,...
228991,68523,1967-12-29,0.481481
228992,76151,1967-12-29,-0.072165
228993,84751,1967-12-29,<NA>
228994,84911,1967-12-29,0.057971


In [7]:
msf_5767['date'] = pd.to_datetime(msf_5767['date'])

In [8]:
msf_5767 = msf_5767.sort_values(['permno', 'date']).reset_index(drop=True)

In [10]:
# explicitly check for missing months per permno before lagging
msf_5767['month'] = msf_5767['date'].dt.to_period('M')
msf_5767

,permno,date,ret,month
0,10006,1956-01-31,-0.04428,1956-01
1,10006,1956-02-29,-0.001931,1956-02
2,10006,1956-03-29,0.041257,1956-03
3,10006,1956-04-30,-0.026415,1956-04
4,10006,1956-05-31,-0.054264,1956-05
...,...,...,...,...
228991,86239,1967-08-31,-0.029412,1967-08
228992,86239,1967-09-29,-0.099394,1967-09
228993,86239,1967-10-31,-0.102041,1967-10
228994,86239,1967-11-30,0.037879,1967-11


In [11]:
len(msf_5767)

228996

In [12]:
msf_5767 = msf_5767.sort_values(['permno', 'date']).reset_index(drop=True)


In [13]:
msf_5767['mom1m'] = (
    msf_5767
    .groupby('permno')['ret']
    .shift(1)
)

In [14]:
msf_5767 = msf_5767.sort_values(['permno', 'date']).reset_index(drop=True)

In [15]:
import numpy as np

msf_5767['mom6m'] = (
    msf_5767
    .groupby('permno')['ret']
    .apply(lambda s: (1.0 + s.shift(2)).rolling(5, min_periods=5).apply(np.prod, raw=True) - 1.0)
    .reset_index(level=0, drop=True)
)


In [16]:
msf_5767['mom12m'] = (
    msf_5767
    .groupby('permno')['ret']
    .apply(lambda s: (1.0+s.shift(2)).rolling(11, min_periods=11).apply(np.prod, raw=True) - 1.0)
    .reset_index(level=0, drop=True)
)

In [18]:
msf_5767_mom = msf_5767.dropna(
    subset=['mom1m', 'mom6m', 'mom12m']
).reset_index(drop=True)


In [19]:
msf_5767_mom.sort_values(['date','permno']).reset_index(drop=True)

,permno,date,ret,month,mom1m,mom6m,mom12m
0,10006,1957-01-31,0.064378,1957-01,0.044843,-0.059517,-0.120199
1,10014,1957-01-31,0.095238,1957-01,-0.086957,-0.115385,-0.041668
2,10022,1957-01-31,0.102041,1957-01,-0.060377,-0.039552,-0.108753
3,10030,1957-01-31,-0.047091,1957-01,0.044633,0.050470,0.133651
4,10057,1957-01-31,-0.090062,1957-01,0.086667,0.055248,0.182845
...,...,...,...,...,...,...,...
189075,68523,1967-12-29,0.481481,1967-12,0.08,0.056338,0.500000
189076,76151,1967-12-29,-0.072165,1967-12,0.114943,0.279412,1.071428
189077,84751,1967-12-29,<NA>,1967-12,-0.005839,0.132118,0.467658
189078,84911,1967-12-29,0.057971,1967-12,-0.067568,1.387096,2.700000


## bid ask spread

In [20]:
db = wrds.Connection()

Loading library list...
Done


In [21]:
query = """
select
    permno,
    date,
    ret,
    prc,
    vol,
    shrout,
    askhi,
    bidlo
from crsp.dsf
WHERE date >= '1956-01-01'
  AND date <= '1967-12-31'
"""

In [22]:
dsf = db.raw_sql(query)
db.close()

In [27]:
dsf['date'] = pd.to_datetime(dsf['date'])

In [28]:
dsf['yr'] = dsf['date'].dt.year
dsf['month'] = dsf['date'].dt.month

In [30]:
def compute_monthly_baspread(df):
    """
    df: daily CRSP data for a single permno-month
        must contain askhi and bidlo
    """
    spread = (df['askhi'] - df['bidlo']) / ((df['askhi'] + df['bidlo']) / 2)
    return spread.mean()

In [47]:
baspread = (
    dsf
    .groupby(['permno', 'yr', 'month'])
    .apply(compute_monthly_baspread)
    .reset_index(name='baspread')
)

/var/folders/vs/j67b0sxj4nj63kxt4lvn9ctw0000gn/T/ipykernel_22775/3501628674.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_monthly_baspread)


In [56]:
baspread['date'] = pd.to_datetime(
    dict(year=baspread['yr'], month=baspread['month'], day=1)
)
baspread

,permno,yr,month,baspread,date
0,10006,1956,1,0.013017,1956-01-01
1,10006,1956,2,0.010517,1956-02-01
2,10006,1956,3,0.00956,1956-03-01
3,10006,1956,4,0.011858,1956-04-01
4,10006,1956,5,0.01607,1956-05-01
...,...,...,...,...,...
227228,86239,1967,8,0.026636,1967-08-01
227229,86239,1967,9,0.02228,1967-09-01
227230,86239,1967,10,0.02584,1967-10-01
227231,86239,1967,11,0.028309,1967-11-01


In [57]:
# 把 msf 的 date 统一转成月初
msf_5767_mom['date_m'] = msf_5767_mom['date'].values.astype('datetime64[M]')

# 构造 t-1 月
msf_5767_mom['date_lag1'] = msf_5767_mom['date_m'] - pd.offsets.MonthBegin(1)

In [58]:
msf_5767_mom = msf_5767_mom.merge(
    baspread[['permno', 'date', 'baspread']],
    left_on=['permno', 'date_lag1'],
    right_on=['permno', 'date'],
    how='left'
)

In [63]:
msf_5767_mom.columns

Index(['permno', 'date_x', 'ret', 'month', 'mom1m', 'mom6m', 'mom12m',
       'date_m', 'date_lag1', 'date_y', 'baspread'],
      dtype='object')

In [64]:
msf_5767_mom.sort_values(['date_x','permno']).reset_index(drop=True)

,permno,date_x,ret,month,mom1m,mom6m,mom12m,date_m,date_lag1,date_y,baspread
0,10006,1957-01-31,0.064378,1957-01,0.044843,-0.059517,-0.120199,1957-01-01,1956-12-01,1956-12-01,0.013234
1,10014,1957-01-31,0.095238,1957-01,-0.086957,-0.115385,-0.041668,1957-01-01,1956-12-01,1956-12-01,0.033305
2,10022,1957-01-31,0.102041,1957-01,-0.060377,-0.039552,-0.108753,1957-01-01,1956-12-01,1956-12-01,0.016023
3,10030,1957-01-31,-0.047091,1957-01,0.044633,0.050470,0.133651,1957-01-01,1956-12-01,1956-12-01,0.015295
4,10057,1957-01-31,-0.090062,1957-01,0.086667,0.055248,0.182845,1957-01-01,1956-12-01,1956-12-01,0.005954
...,...,...,...,...,...,...,...,...,...,...,...
189075,68523,1967-12-29,0.481481,1967-12,0.08,0.056338,0.500000,1967-12-01,1967-11-01,1967-11-01,0.042217
189076,76151,1967-12-29,-0.072165,1967-12,0.114943,0.279412,1.071428,1967-12-01,1967-11-01,1967-11-01,0.013458
189077,84751,1967-12-29,<NA>,1967-12,-0.005839,0.132118,0.467658,1967-12-01,1967-11-01,1967-11-01,0.040715
189078,84911,1967-12-29,0.057971,1967-12,-0.067568,1.387096,2.700000,1967-12-01,1967-11-01,1967-11-01,0.053848
